In [ ]:
from serial.tools import list_ports
ports = list(list_ports.comports())

if not ports:
    print("No serial ports found. Check the USB cable, board connection, and drivers.")
else:
    for p in ports:
        print(f"{p.device:20s} | {p.description} | {p.hwid}")

PORT = [p for p in ports if p.description == "Nano 33 BLE"][0].device
BAUD_RATE = 9600
READ_TIMEOUT_SECONDS = 1

print(f"Using port: {PORT}")


## Data format

From the `main.cpp` file, IMU data is sent to serial with the following format:

```cpp
  snprintf(line,sizeof(line),"%.3f|%.3f", ax, average);
```

In [ ]:
import time
from datetime import datetime

SENSOR_DATA = ["accel_x", "avg_accel_x"]
def parse_sensor_line(line: str): 
    line = line.strip()
    numbers = [float(x) for x in line.split("|")]

    data_point = {
        "timestamp": datetime.now().isoformat(timespec="milliseconds"),
        "raw": line,
        "accel_x": 0.0,
        "avg_accel_x": 0.0,
    }

    if len(numbers) >= 2:
        for name, value in zip(SENSOR_DATA, numbers[:2]):
            data_point[name] = value
    return data_point

# Quick parser test
test_line = "0.045|0.001"
parse_sensor_line(test_line)


In [ ]:
import pandas as pd
import serial

def collect_sensor_data(port: str=PORT, baud_rate: int=BAUD_RATE, seconds: int=10):
    points = []
    print(f"Opening {port} at {baud_rate} baud...")

    with serial.Serial(port, baud_rate, timeout=READ_TIMEOUT_SECONDS) as ser:
        time.sleep(2)
        ser.reset_input_buffer()

        print("Collecting data. Press the stop button in Jupyter to interrupt.")
        start = time.time()

        while time.time() - start < seconds:
            raw = ser.readline()
            if not raw:
                continue

            line = raw.decode("utf-8", errors="replace").strip()
            if not line:
                continue

            data_point = parse_sensor_line(line)
            data_point["elapsed_seconds"] = time.time() - start
            points.append(data_point)

    df = pd.DataFrame(points)
    print(f"Collected {len(df)} data points.")
    return df

df = collect_sensor_data(seconds=10)
df.head()


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

columns=("accel_x", "avg_accel_x")
plt.figure(figsize=(10, 5))
for col in columns:
    plt.plot(df["elapsed_seconds"], df[col], label=col)
plt.xlabel("Elapsed time (seconds)")
plt.ylabel("Sensor value")
plt.title("Live Nano 33 BLE Sensor Stream")
plt.legend(loc="upper right")
plt.grid(True)
plt.show()


In [ ]:
df.describe()


In [ ]:
import numpy as np

# 1. Define the list of conditional checks
conditions = [df["avg_accel_x"] >= 0.1, df["avg_accel_x"] < 0.1]

# 2. Define the corresponding labels for those conditions
labels = ["Y", "N"]

# 3. Apply numpy select
df["Label"] = np.select(conditions, labels, default="Unknown")
df.head(50)

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
output_file = DATA_DIR / "nano33ble_sensor_capture.csv"

df_cleaned = df.iloc[10:-10]
df_cleaned.to_csv(output_file, index=False)

print(f"Saved {len(df_cleaned)} rows to {output_file}")
